<br><br>
<h1><font color="#B30033" size=5>Intelligent Systems - Academic year 2026-2027</font></h1>

<h1><font color="#B30033" size=5>Lab 1: State space search</font></h1>

<br>
<div style="text-align: left">
<font color="#4E70BE" size=3>Teaching staff:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Mª Julia Flores Gallego (Julia.Flores@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Antonio Gámez Martín (Jose.Gamez@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Ismael García Varea (Ismael.Garcia@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Roldán Gómez (Jose.Roldan@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Bastián Andrés Troncoso Retamales (Bastian.Troncoso@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Teresa Valero Díaz (Teresa.Valero@uclm.es)</font><br></li>
</ul>
</div>

<div style="text-align: left">
<font color="#4E70BE" size=3>Submitted by:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Full name, e-mail</font><br></li>
  <li><font color="#4E70BE" size=3>Full name, e-mail</font><br></li>
</ul>
</div>

--------------------


## 1. Introduction

In this lab you will put to work the state space search strategies of **units 2 and 3**.
The setting is a **multi-level space station**: a maintenance robot has to move from one
point of the station to another using as little energy as possible.

The station is the **common environment of the three labs** of the course. What you build
here you will use again in Lab 3, where the question is no longer *which path do I take*
but *in which order do I visit several points*; and you will meet the same station again
in Lab 2, except that there the robot has no map and has to learn how to move.

### 1.1. Objectives

* Formalise a real problem as **state space search**.
* Implement **a single best-first search function**, parameterised by $f(n)$, and
  instantiate it as four strategies: breadth-first, uniform cost, greedy and $A^*$.
* Handle **repeated states** correctly, including the case that causes most errors:
  reaching an already visited state along a cheaper path.
* **Measure**: compare the strategies on instances of different sizes using the five
  counters of the course, and reason about what you observe.
* Study the effect of **heuristic quality** and see in practice what it means for one
  heuristic to dominate another.


## 2. Problem description

The station has several **levels**, and each level is a **grid**. A position is a triple
`[level, row, column]`.

Every cell has an **occupancy cost**:

| Value | Meaning |
|:--:|:--|
| `0` | **Not traversable**: bulkhead, hull or sealed area |
| `1` | Clear corridor |
| `2` | Low-gravity bay |
| `3` | Service duct |

**Horizontal movement.** Four directions within the same level (north, east, south,
west). No diagonals. The cost of moving from a cell $u$ to an adjacent cell $v$ is

$$c(u,v) = \max\big(\text{cost}(u),\ \text{cost}(v)\big)$$

that is, *a step costs as much as the worse of its two cells*. Note that this makes the
cost **symmetric**: going and coming back cost the same.

**Vertical movement.** Only through **hatches**, which are declared explicitly in the
instance. A hatch connects the **same (row, column)** between **two consecutive levels**,
and has its own cost, the same in both directions. A cell with a hatch can be crossed
horizontally like any other.

### 2.1. Formalisation

Before writing any code, be clear about the elements of the search problem. Remember the
distinction made in unit 2: **states have no parent, no depth and no cost; nodes do**. The
same state may be represented by several different nodes of the search tree.

| Element | In this problem |
|:--|:--|
| States | Traversable positions `[level, row, column]` |
| Initial state | `estacion.origen` |
| Actions | `NORTE`, `ESTE`, `SUR`, `OESTE`, `SUBIR`, `BAJAR` |
| Transition model | The adjacent position in that direction, if traversable |
| Goal test | The position is `estacion.destino` |
| Action cost | $\max$ of the costs of the two cells, or the cost of the hatch |

> The identifiers of the support package are in Spanish, the same ones the other groups
> use: `origen` (origin), `destino` (destination), `sucesores` (successors), `coste`
> (cost), `camino` (path), `frontera` (frontier), `alcanzados` (reached).


## 3. Working on the lab

You will work on a **catalogue of ready-made instances**, in `catalogo/pr1/`. They range
from small 2-level stations with a few dozen positions up to a 5-level one with more than
22,000. Your algorithms have to work on all of them: anything that scales badly will show.

The stations come in two styles, and the `H` suffix in the name tells you which is which:

* **Open-plan** (`PR1-M-01`, `PR1-L-01`…): wide halls, with the bulkheads almost always
  hugging the hull. What makes the path vary is mostly the cost-2 and cost-3 areas.
* **With rooms** (`PR1-MH-01`, `PR1-LH-01`, `PR1-XLH-01`…): rectangular compartments
  separated by bulkheads and connected by doors, which force detours.

Each `MH` or `LH` instance comes from the same seed and has the same size as its open-plan
twin (`PR1-MH-01` with `PR1-M-01`, and so on), so you can compare the same station with
and without compartments.

Assessment and the lab exam use **new instances** in the same format, produced by the same
generator.

### 3.1. The `ssii_estacion` package

You are given a package with all the infrastructure: instance loading and validation,
successor generation, the priority queue, the set of reached states, the counters, the
JSON dump of solutions and the visualiser.

**You must not modify it, and you do not need to.** Your job is the search loop, not the
plumbing. There is also a strong reason for this: the five counters are extremely
sensitive to the order in which successors are generated and to how ties are broken. Since
the package fixes both, your numbers and ours are comparable, and marking can be
automatic.

The two rules the package fixes, which you should know about:

1. **Canonical successor order**: `NORTE`, `ESTE`, `SUR`, `OESTE`, `SUBIR`, `BAJAR`.
2. **FIFO tie-breaking**: between two nodes with the same $f$, the one inserted first
   comes out first.


## 4. Work plan

### 4.1. One search, four strategies

Instead of writing four algorithms you are going to write **one**: best-first graph
search, taking as a parameter the function $f$ that orders the frontier. The four
strategies are instances of it:

| Strategy | $f(n)$ |
|:--|:--|
| Breadth-first | $\text{depth}(n)$ |
| Uniform cost | $g(n)$ |
| Greedy | $h(n)$ |
| A$^*$ | $g(n) + h(n)$ |

This is not a shortcut to save work: it is the `BEST-FIRST-SEARCH(problem, f)` framework
Russell and Norvig use throughout the chapter. Writing it this way makes it obvious that
the four strategies differ **only** in how they prioritise the frontier.

### 4.2. Two warnings about repeated states

These are the two most common mistakes, and both affect correctness, not efficiency.

**Goal test on removal, not on generation.** Check whether a node is the goal when you
**take it out** of the frontier, not when you create it. If you test on generation,
uniform cost and A$^*$ stop being optimal: the frontier may still hold a cheaper path to
the same destination that has not come out yet.

**A repeated state is not always discarded.** If you reach a state you had already
reached, but **along a cheaper path**, you have to keep the new one. Discarding it without
looking at the cost breaks optimality. That is what `Cerrados.mejora(estado, g)` is for: it
returns `True` only if the state is new or if you have improved its best known cost.

### 4.3. Heuristics

You are given two, both admissible and consistent:

$$h_1(p) = |\Delta r| + |\Delta c| + |\Delta l| \qquad
  h_2(p) = |\Delta r| + |\Delta c| + |\Delta l| \cdot c_{\min}$$

where $c_{\min}$ is the cost of the cheapest hatch in the station. $h_2$ **dominates**
$h_1$: it is never smaller. Part of your job is to check whether that dominance always
translates into fewer expanded nodes.

### 4.4. Experimental study

Once the algorithms work, run the four strategies over the whole catalogue and build a
comparative table. Do not just paste it: **interpret it**. Section 8 lists the questions
your report has to answer.

### 4.5. Report

Besides the notebook, you submit a report with the formalisation of the problem, the
implementation decisions, the experimental study with its plots, and the conclusions. The
critical analysis of the results is the part that weighs most.

### 4.6. Session plan

Lab 1 has four lab sessions. A reasonable way to split them:

1. **Session 1**: the environment and the problem. Run section 6: load `PR1-S-01`, look at
   its successors in canonical order, compare the two heuristics at the origin and draw the
   station. Get familiar with `Nodo`, `Frontera`, `Cerrados` and `Contadores` (section 6.2).
   Formalise the problem —states, actions, costs, goal—, which is the first part of the
   report, and start `primero_el_mejor` (7.1).
2. **Session 2**: a correct search. Finish `primero_el_mejor` keeping in mind the two
   warnings in 4.2, write the four strategies (7.2) and check that on `PR1-S-01` uniform
   cost and $A^*$ give cost 18 and greedy gives 26. Pass the public tests
   (`python3 -m pytest PR1/pruebas -q`) before leaving the lab.
3. **Session 3**: the whole catalogue. Generate the solutions with the cell in 7.3, build
   the table and plots of section 8 and keep an eye on time and memory in `PR1-XLH-01`. If
   a large instance takes much longer than the rest, the problem is almost always in how
   the frontier or the closed set is handled, not in the size.
4. **Session 4**: analysis and report. Answer the questions in 8.3 with your data,
   including 7 and 8, which ask you to change the code and measure again (do it on a copy
   and leave the correct version in the notebook you submit). Finish the report and review
   the submission. If you follow non-continuous assessment, start the extension in
   section 9 here.

Always debug on `PR1-S-01`: 86 positions and a single hatch. If something goes wrong there
you can see it in the drawing; on an L instance you cannot.


## 5. Submission and assessment

The work is done **in pairs** (individually is also accepted; groups of three or more are
not allowed). You submit **this very notebook**, completed, together with the JSON
solution files produced in section 7.

Remember that lab work is assessed with two components:

1. **25 %** — code submission and performance on a battery of automatic tests.
2. **75 %** — lab exam, held in the last week of class.

A **minimum of 4 out of 10** in the weighted average is required to pass or compensate
this part of the course.

If you follow **non-continuous assessment**, you also submit the extension in section 9:
you have more time to solve and hand in the lab. Students under continuous assessment do
not have to submit it.

### 5.1. Automatic marking

Solutions are checked automatically from the JSON files. **Your path is not compared with
ours**: a grid has many optimal paths and they are all equally valid. What is checked is:

* that the path starts at the origin, ends at the destination, and that every step joins
  positions that really are adjacent and traversable;
* that the declared cost matches the cost recomputed over the path;
* that the cost is optimal when the algorithm guarantees it;
* that the five counters are consistent with each other and match the reference ones.

That is why the canonical successor order and the package structures matter.

Some of those checks are in your hands. From the lab folder:

If you did not followed the videos, and installed the environment with the README version, this is the way:

```bash
pip install pytest
python3 -m pytest PR1/pruebas -q
```

If you followed the videos and have miniconda installed:
```bash
python -m pytest -q
```

These are the **public tests**: the contract of your functions and the edge cases (origin
equal to destination, unreachable destination, compulsory hatch). Passing them is the
minimum, not the mark: marking adds further tests you do not see, on instances generated
on the spot and on the five counters. The details are in `PR1/pruebas/README.md`.

### 5.2. On the use of generative AI tools

This problem is one an assistant solves in a single go, so let us be explicit: **handing in
the work is a declaration that it is your own**, and in the lab exam —worth 75 % of this
part— you will have to work on this same code without help. The course policy described in
unit 0 applies.


## 6. The environment at work

We start by checking that everything loads and seeing how the station is queried.

In [18]:
# The ssii_estacion package sits in the folder above, next to the catalogue.
import sys, os
sys.path.append(os.path.abspath(".."))

from ssii_estacion import (
    cargar_instancia, Pos,
    Frontera, Cerrados, Contadores, Nodo, reconstruir_camino,
    solucion_pr1, guardar_solucion, validar_solucion_pr1,
)
from ssii_estacion.vis import dibujar

print("package loaded successfully")

In [19]:
# A small instance, to see how everything works
est = cargar_instancia("../catalogo/pr1/PR1-S-01.json")

print(est)
print("origin     :", est.origen)
print("destination:", est.destino)
print("cost of the cheapest hatch:", est.coste_min_conexion)

# Successors of a position, ALWAYS in canonical order
print("\nSuccessors of the origin:")
for accion, destino, coste in est.sucesores(est.origen):
    print(f"  {accion:6s} -> {destino}  cost {coste}")

# The two heuristics
print("\nh1(origin) =", est.h_manhattan(est.origen, est.destino))
print("h2(origin) =", est.h_manhattan_conexiones(est.origen, est.destino))

In [20]:
# Visualisation: the levels are drawn in a row, with the hatches marked
dibujar(est, puntos={"origin": est.origen, "destination": est.destino},
        titulo=est.identificador);

### 6.1. The two station styles

The instance above is a small one. These two have the same size and come from the same
seed, and differ only in how the space is divided up: this is the comparison asked for in
question 3 of section 8.

As you look at them, think about **where the path will have to go** in each one, and where
the hatches are (the small squares): in the open-plan one almost any route is possible and
what makes it expensive is crossing the orange areas; in the one with rooms, every route
between two compartments goes through a door.

In [21]:
# The same station, with and without compartments
for nombre in ("PR1-M-01", "PR1-MH-01"):
    e = cargar_instancia(f"../catalogo/pr1/{nombre}.json")
    estilo = e.metadatos.get("estilo", "diafana")
    print(f"{nombre}: {e.n_transitables()} traversable positions, style {estilo}")
    dibujar(e, puntos={"origin": e.origen, "destination": e.destino},
            titulo=f"{nombre} — {estilo}");

### 6.2. API summary

What you will need:

```python
est.sucesores(p)                      # [(action, target, cost)] in canonical order
est.es_transitable(p)                 # bool: is it traversable?
est.coste_celda(p)                    # 0 if not traversable
est.coste_transicion(a, b)            # cost if adjacent, None otherwise
est.h_manhattan(p, objetivo)          # h1
est.h_manhattan_conexiones(p, objetivo)  # h2
est.origen, est.destino               # origin and destination
```

And the search structures:

```python
cont       = Contadores()             # the five counters
frontera   = Frontera(cont)           # priority queue with FIFO tie-breaking
alcanzados = Cerrados(cont)           # best known cost per state

frontera.insertar(nodo, f)            # insert with priority f
nodo = frontera.extraer()             # remove the smallest f  (counts as explored)
frontera.vacia()                      # is it empty?

alcanzados.mejora(estado, g)          # True if new, or if g improves what is known
alcanzados.coste(estado)              # best known cost, or None

Nodo(estado=..., padre=..., accion=..., coste=..., profundidad=...)
reconstruir_camino(nodo)              # -> (list of positions, list of actions)
```

The five counters, with their exact definition:

| Counter | Definition |
|:--|:--|
| `generados` | Nodes created, including the root node |
| `explorados` | Nodes **removed** from the frontier |
| `expandidos` | Of those explored, the ones whose successors are generated |
| `max_frontera` | Maximum size of the open list |
| `max_alcanzados` | Maximum size of the closed set |

`Frontera` and `Cerrados` update `explorados`, `max_frontera` and `max_alcanzados` by
themselves. The other two —`generados` and `expandidos`— **you have to increment
yourself**, in the right place. It must always hold that
`expandidos <= explorados <= generados`.


## 7. Implementation

### 7.1. The search loop

This is the only thing you write, and everything else follows from it. Read the skeleton
carefully: each `TODO` marks one decision, and the comments say where the two mistakes of
section 4.2 hide.


```python
function GRAPH-SEARCH(problem, strategy) returns a solution, or failure
    initialize the search tree using the initial state of problem
    loop do
        if there are no candidates for expansion then 
            return failure
        choose a leaf node for expansion according to strategy
        if the state within the node has NOT been visited yet AND the node contains a goal state then 
            return the corresponding solution
        else 
            add the node to the set of closed/visited 
            expand the node
            add the resulting nodes to the search tree
    end
```
With our structure and considering the cost before discarding:

```python
function GRAPH-SEARCH_WITH_BEST_FIRST(problem, strategy) returns a solution, or failure
(1) initialize frontier with the initial state of problem
(2) initialize reached with the initial state and cost 0
(3) while frontier is not empty do
(4)     choose a leaf node for expansion according to strategy
(5)     if the node contains a goal state then              (!) WARNING 1
            return the corresponding solution
(6)     if the node cost is worse than reached cost then    (!) WARNING 2
            skip to next iteration
(7)     expand the node
(8)     for each resulting child node:
(9)         if child state has NOT been visited yet OR child has better cost then (!) WARNING 2
(10)            update reached with child state and cost
(11)            add child node to frontier
(12) return failure
```

In [22]:
def primero_el_mejor(estacion, origen, destino, f, h=None):
    """Best-first graph search.

    Parameters
    ----------
    f : function (node, h) -> priority
        This is what tells one strategy from another.
    h : function (position) -> estimated cost to the destination
        Only the informed strategies use it.

    Returns a dictionary with: encontrada, camino, acciones, coste, contadores.
    """
    h = h or (lambda _p: 0) 

    """ 
    Lambda function on an assignment.
    Line: h = h or (lambda _p: 0) is the same as this:
    if h is None:
        def h(_p):  <- now h is a function that always returns 0
            return 0
    
    """

    cont = Contadores() #(1)
    frontera = Frontera(cont) # (1)
    alcanzados = Cerrados(cont) # (2)

    raiz = Nodo(estado=origen, coste=0, profundidad=0) # (1)
    cont.generados += 1
    alcanzados.mejora(origen, 0)
    frontera.insertar(raiz, f(raiz, h))

    while not frontera.vacia(): # (3)
        nodo = frontera.extraer() # (4) -> extraer() returns the node with less f(), that means, best first
            
        if nodo.estado == destino:
            camino, acciones = reconstruir_camino(nodo)
            return {"encontrada": True, "camino": camino, "acciones": acciones, "coste": nodo.coste,
                    "contadores": cont}
            
        mejor_coste = alcanzados.coste(nodo.estado)
        if mejor_coste is not None and nodo.coste > mejor_coste:
            continue
        
        cont.expandidos += 1
        # Creating hijo
        for accion, sucesor, coste_paso in estacion.sucesores(nodo.estado):
            coste_hijo = nodo.coste + coste_paso
            hijo = Nodo(
                estado=sucesor,
                padre=nodo,
                accion=accion,
                coste=coste_hijo,
                profundidad=nodo.profundidad + 1,
            )
            cont.generados += 1
            # Adding hijo
            if alcanzados.mejora(sucesor, coste_hijo):
                frontera.insertar(hijo, f(hijo, h))

    return {"encontrada": False, "camino": [], "acciones": [], "coste": None,
            "contadores": cont} # (12)

### 7.2. The four strategies

Now the four strategies are four lines. Define each one by passing the corresponding $f$.


In [23]:
def anchura(estacion, origen, destino):
    def prioridad(nodo, h):
        return nodo.profundidad
    return primero_el_mejor(estacion, origen, destino, prioridad)

def coste_uniforme(estacion, origen, destino):
    def prioridad(nodo, h):
        return nodo.coste
    return primero_el_mejor(estacion, origen, destino, prioridad)

def voraz(estacion, origen, destino, h):
    def prioridad(nodo, h):
        return h(nodo.estado)
    return primero_el_mejor(estacion, origen, destino, prioridad, h)

def a_estrella(estacion, origen, destino, h):
    def prioridad(nodo, h):
        return nodo.coste + h(nodo.estado)
    return primero_el_mejor(estacion, origen, destino, prioridad, h)

In [24]:
# A quick check on the small instance.
# The expected values for PR1-S-01 are:
#   optimal cost = 18   |   cost of the greedy search = 26
#   (breadth-first gives 22: the path with the fewest steps is not the cheapest)
r = coste_uniforme(est, est.origen, est.destino)
print("uniform cost -> cost", r["coste"], "|", r["contadores"].como_dict())

h2 = lambda p: est.h_manhattan_conexiones(p, est.destino)
r = a_estrella(est, est.origen, est.destino, h2)
print("A* with h2   -> cost", r["coste"], "|", r["contadores"].como_dict())

dibujar(est, camino=r["camino"],
        puntos={"origin": est.origen, "destination": est.destino},
        titulo=f"A* with h2 — cost {r['coste']}");

### 7.3. Producing the solution files

This cell writes the JSON files you submit and the marking system checks. Do not modify
it: only change the list of authors.


In [25]:
AUTORES = ["Full name <e-mail@alu.uclm.es>",
           "Full name <e-mail@alu.uclm.es>"]

INSTANCIAS = ["PR1-S-01", "PR1-S-02",
              "PR1-M-01", "PR1-M-02", "PR1-M-03", "PR1-MH-01", "PR1-MH-02", "PR1-MH-03",
              "PR1-L-01", "PR1-L-02", "PR1-LH-01", "PR1-LH-02",
              "PR1-XLH-01"]

import time, os

def resolver_y_exportar(nombre_instancia, carpeta="soluciones"):
    est = cargar_instancia(f"../catalogo/pr1/{nombre_instancia}.json")
    h1 = lambda p: est.h_manhattan(p, est.destino)
    h2 = lambda p: est.h_manhattan_conexiones(p, est.destino)

    ejecuciones = {
        "anchura":        (lambda: anchura(est, est.origen, est.destino), None),
        "coste_uniforme": (lambda: coste_uniforme(est, est.origen, est.destino), None),
        "voraz_h2":       (lambda: voraz(est, est.origen, est.destino, h2), "h2"),
        "a_estrella_h1":  (lambda: a_estrella(est, est.origen, est.destino, h1), "h1"),
        "a_estrella_h2":  (lambda: a_estrella(est, est.origen, est.destino, h2), "h2"),
    }

    filas = []
    for algoritmo, (fn, heur) in ejecuciones.items():
        t0 = time.perf_counter()
        r = fn()
        t = time.perf_counter() - t0
        doc = solucion_pr1(
            instancia_id=est.identificador, algoritmo=algoritmo, autores=AUTORES,
            camino=r["camino"], acciones=r["acciones"], coste=r["coste"],
            contadores=r["contadores"], heuristica=heur, tiempo_s=t,
            encontrada=r["encontrada"],
        )
        problemas = validar_solucion_pr1(est, doc)
        if problemas:
            print(f"  !! {nombre_instancia} / {algoritmo}: {problemas}")
        guardar_solucion(doc, os.path.join(carpeta, f"{nombre_instancia}.{algoritmo}.json"))
        filas.append({"instancia": nombre_instancia, "algoritmo": algoritmo,
                      "estilo": est.metadatos.get("estilo", "diafana"),
                      "estados": est.n_transitables(),
                      "coste": r["coste"], "tiempo_s": round(t, 4),
                      **r["contadores"].como_dict()})
    return filas

resultados = []
for nombre in INSTANCIAS:
    print("solving", nombre, "...")
    resultados += resolver_y_exportar(nombre)
print("\ndone: solution files written to ./soluciones/")

## 8. Experimental study

### 8.1. Comparative table


In [26]:
import pandas as pd

df = pd.DataFrame(resultados)
tabla = df.pivot_table(index="instancia", columns="algoritmo",
                       values=["coste", "nodos_expandidos"], sort=False)
display(tabla)

### 8.2. Plots

A table of numbers convinces nobody. Plot at least:

* expanded nodes against instance size, for the four strategies;
* the cost of the solution found against the optimal cost;
* the relative saving of A$^*$ over uniform cost, and how it evolves with size.


In [27]:
import matplotlib.pyplot as plt

# Colour per algorithm; circle = open-plan station, square = with rooms
marcas = {"diafana": "o", "habitaciones": "s"}
colores = dict(zip(df["algoritmo"].unique(), plt.rcParams["axes.prop_cycle"].by_key()["color"]))

fig, ax = plt.subplots(figsize=(7, 4.5))
for (alg, estilo), g in df.groupby(["algoritmo", "estilo"]):
    ax.scatter(g["estados"], g["nodos_expandidos"], marker=marcas[estilo],
               color=colores[alg], alpha=.8, label=f"{alg} ({estilo})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("traversable positions of the station")
ax.set_ylabel("expanded nodes")
ax.set_title("Search effort by strategy")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
plt.tight_layout()

### 8.3. Analysis questions

These are the questions your report has to answer, with your data in front of you. Do not
look for the answer in the lecture notes: find it in your table and explain it.

1. **Breadth-first search returns paths with a higher cost than uniform cost in several
   instances.** Why? What is each one optimising, exactly? Find an instance in your table
   where it happens and explain what is special about it.

2. **How much does $A^*$ save over uniform cost, and does that advantage grow or shrink
   with the size of the station?** Give concrete percentages and risk an explanation.

3. **The same station, with and without rooms.** Compare each pair (`PR1-M-01` with
   `PR1-MH-01`, and so on). For each one, compute the ratio $h_2(\text{origin}) / C^*$,
   where $C^*$ is the optimal cost, and the saving of A$^*$ over uniform cost. What
   relation do you see between the two numbers? Why do walls and doors make the heuristic
   worse? And why does greedy search suffer so much in the `XLH` instance?

4. **Greedy search does not find the optimum in any instance of the catalogue.** Why not?
   Is it always faster than the rest? Find a case in your table where it expands *more*
   nodes than uniform cost and try to explain what happened in that station.

5. **$h_2$ dominates $h_1$: it is never smaller.** Check it in your data. Does it *always*
   translate into fewer expanded nodes? What is the difference between "dominating" and
   "being better in every instance"?

6. **Both heuristics underestimate the real cost considerably.** Compute the mean of $h_2$
   and the mean of the real cost in an M instance. Can you think of a better admissible
   heuristic, taking advantage of the fact that you know the cell costs?

7. **Goal test on removal versus on generation.** Change your code to test the goal when
   the node is generated. Do the expanded nodes go down? Is the cost you obtain still
   optimal? Find an instance where the answer changes and explain it.

8. **Repeated states.** Modify `Cerrados` so that it discards any already reached state
   without comparing costs. Which instances stop giving the optimum? Why some and not
   others?

9. **Memory.** Compare `max_frontera` and `max_alcanzados` across strategies. Which is the
   real bottleneck when solving the `XLH` instance?

## 9. Extensions

**Non-continuous assessment: this section is compulsory.** You have more time to solve and
hand in the lab, so your submission also includes what is asked here. **Continuous
assessment: it is optional**, and whatever you submit counts towards the code mark.

Both reuse the same `primero_el_mejor` loop of section 7: change $f$, or change the
traversal, but not the package structures. For each one, export the solution files as in
section 7.3 (with the algorithm name given below) and add a table and a paragraph with
what you observe.

* **Weighted $A^*$.** Use $f(n) = g(n) + W\cdot h(n)$ with $W > 1$. You lose the
  optimality guarantee, but you get a bound: the cost never exceeds $W$ times the optimum.
  Run it with $W \in \{1;\ 1.5;\ 2;\ 3\}$ over the whole catalogue, check that the bound
  holds in every case, and plot the cost obtained against the expanded nodes. Beyond which
  $W$ does it stop paying off? Algorithm: `a_estrella_ponderada_W`.
* **A better heuristic.** $h_1$ and $h_2$ ignore cell costs entirely. Design an admissible
  heuristic that takes them into account, **prove that it is admissible** (and say whether
  it is consistent), and compare it with $h_2$ in expanded nodes over the catalogue. Check
  as well that it dominates $h_2$ on the states you visit. Algorithm: `a_estrella_h3`.

## 10. References

* S. Russell and P. Norvig. *Artificial Intelligence: A Modern Approach*, 4th ed., Pearson,
  2021. Chapter 3, sections 3.1 to 3.6.
* Lecture slides of units 2 and 3.
